<a href="https://colab.research.google.com/github/volgasezen/is584/blob/main/Lab 8/1_finetuning_bert_huggingface.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<h1 style="margin-bottom:0">IS 584: Deep Learning for Text Analytics</center></h1>
<h3 style="margin-top:0">Lab 8: Fine-tuning DistilBert for Yelp Review Classification</center></h2>
<i>

</i></center>

This lab is adapted from huggingface tutorials. Regular version includes videos and sections that use huggingface trainer class. We will train our model with pure pytorch in this session, focusing more on huggingface libraries `transformers` and `datasets` to download open-weight models and popular datasets.

# Fine-tune a pretrained model

There are significant benefits to using a pretrained model. It reduces computation costs, your carbon footprint, and allows you to use state-of-the-art models without having to train one from scratch. 🤗 Transformers provides access to thousands of pretrained models for a wide range of tasks. When you use a pretrained model, you train it on a dataset specific to your task. This is known as fine-tuning, an incredibly powerful training technique.

In this tutorial, you will fine-tune a pretrained model with PyTorch.

## Prepare a dataset

Before you can fine-tune a pretrained model, download a dataset and prepare it for training. The previous tutorial showed you how to process data for training, and now you get an opportunity to put those skills to the test!

Begin by loading the [Yelp Reviews](https://huggingface.co/datasets/yelp_review_full) dataset:

In [ ]:
from datasets import load_dataset

train, test = load_dataset("yelp_review_full", split=['train[:10000]', 'test[:1000]'])
train

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Dataset({
    features: ['label', 'text'],
    num_rows: 10000
})

As you now know, you need a tokenizer to process the text and include a padding and truncation strategy to handle any variable sequence lengths. To process your dataset in one step, use 🤗 Datasets [`map`](https://huggingface.co/docs/datasets/process.html#map) method to apply a preprocessing function over the entire dataset:

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained("distilbert-base-cased")

def tokenize_function(examples):
    return tokenizer(examples["text"], truncation=True)

train = train.map(tokenize_function, batched=True)
test = test.map(tokenize_function, batched=True)

Map:   0%|          | 0/1000 [00:00<?, ? examples/s]

## Train in native PyTorch

[Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer) takes care of the training loop and allows you to fine-tune a model in a single line of code. For users who prefer to write their own training loop, you can also fine-tune a 🤗 Transformers model in native PyTorch.

Next, manually postprocess `tokenized_dataset` to prepare it for training.

1. Remove the `text` column because the model does not accept raw text as an input

2. Rename the `label` column to `labels` because the model expects the argument to be named `labels`

3. Set the format of the dataset to return PyTorch tensors instead of lists

In [ ]:
train = train.remove_columns(["text"])
train = train.rename_column("label", "labels")
train.set_format("torch")

test = test.remove_columns(["text"])
test = test.rename_column("label", "labels")
test.set_format("torch")

### DataLoader

Create a `DataLoader` for your training and test datasets so you can iterate over batches of data:

In [ ]:
from torch.utils.data import DataLoader
from transformers import DataCollatorWithPadding

collate_fn = DataCollatorWithPadding(tokenizer=tokenizer)
train_dataloader = DataLoader(train, shuffle=True, batch_size=32, collate_fn=collate_fn)
eval_dataloader = DataLoader(test, batch_size=32, collate_fn=collate_fn)

In [ ]:
a = next(iter(train_dataloader))
a

{'labels': tensor([0, 0, 1, 4, 0, 2, 3, 2, 0, 4, 3, 4, 4, 3, 2, 3, 2, 2, 3, 3, 0, 4, 0, 3,
        2, 3, 0, 0, 4, 1, 0, 2]), 'input_ids': tensor([[  101,  1753,  1363,  ...,     0,     0,     0],
        [  101,  6466,  1377,  ...,     0,     0,     0],
        [  101, 18098,  2094,  ...,     0,     0,     0],
        ...,
        [  101,  1135,   112,  ...,     0,     0,     0],
        [  101,   146,  5165,  ...,     0,     0,     0],
        [  101,  2066,  8756,  ...,     0,     0,     0]]), 'token_type_ids': tensor([[0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        ...,
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0],
        [0, 0, 0,  ..., 0, 0, 0]]), 'attention_mask': tensor([[1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        ...,
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0],
        [1, 1, 1,  ..., 0, 0, 0]])}

Load your model with the number of expected labels:

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained("distilbert-base-cased", num_labels=5)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-cased
Key                     | Status     | 
------------------------+------------+-
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


### Optimizer and learning rate scheduler

Create an optimizer and learning rate scheduler to fine-tune the model. Let's use the [`AdamW`](https://pytorch.org/docs/stable/generated/torch.optim.AdamW.html) optimizer from PyTorch:

In [ ]:
from torch.optim import AdamW

optimizer = AdamW(model.parameters(), lr=5e-5)

Create the default learning rate scheduler from [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer):

In [ ]:
from transformers import get_scheduler

num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    name="linear", optimizer=optimizer, num_warmup_steps=0, num_training_steps=num_training_steps
)

Lastly, specify `device` to use a GPU if you have access to one. Otherwise, training on a CPU may take several hours instead of a couple of minutes.

In [ ]:
import torch

device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

DistilBertForSequenceClassification(
  (distilbert): DistilBertModel(
    (embeddings): Embeddings(
      (word_embeddings): Embedding(28996, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (transformer): Transformer(
      (layer): ModuleList(
        (0-5): 6 x TransformerBlock(
          (attention): DistilBertSelfAttention(
            (q_lin): Linear(in_features=768, out_features=768, bias=True)
            (k_lin): Linear(in_features=768, out_features=768, bias=True)
            (v_lin): Linear(in_features=768, out_features=768, bias=True)
            (out_lin): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (sa_layer_norm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
          (ffn): FFN(
            (dropout): Dropout(p=0.1, inplace=False)


<Tip>

Get free access to a cloud GPU if you don't have one with a hosted notebook like [Colaboratory](https://colab.research.google.com/) or [SageMaker StudioLab](https://studiolab.sagemaker.aws/).

</Tip>

Great, now you are ready to train! 🥳

### Training loop

To keep track of your training progress, use the [tqdm](https://tqdm.github.io/) library to add a progress bar over the number of training steps:

In [ ]:
import torch
from tqdm.auto import tqdm
import evaluate

# 1. Reset your optimizer with a safer learning rate!
# optimizer = AdamW(model.parameters(), lr=5e-5)

num_epochs = 3
metric = evaluate.load("accuracy")

# Set up the overall progress bar
total_steps = len(train_dataloader) * num_epochs
progress_bar = tqdm(range(total_steps), desc="Overall Training")

for epoch in range(num_epochs):
    print(f"\n--- Epoch {epoch + 1}/{num_epochs} ---")

    # --- TRAINING PHASE ---
    model.train()
    total_train_loss = 0
    log_steps = 50 # Print loss every 50 batches

    for step, batch in enumerate(train_dataloader):
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss
        total_train_loss += loss.item()

        loss.backward()
        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

        # Print average loss for the last 'log_steps' batches
        if step % log_steps == 0 and step > 0:
            avg_loss = total_train_loss / log_steps
            print(f"Batch {step:4d}/{len(train_dataloader)} | Avg Loss: {avg_loss:.4f}")
            total_train_loss = 0 # Reset for the next window

    # --- EVALUATION PHASE ---
    print("\nRunning Evaluation...")
    model.eval()

    for batch in eval_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        with torch.no_grad():
            outputs = model(**batch)

        logits = outputs.logits
        predictions = torch.argmax(logits, dim=-1)
        metric.add_batch(predictions=predictions, references=batch["labels"])

    eval_results = metric.compute()
    print(f">>> Epoch {epoch + 1} Accuracy: {eval_results['accuracy']:.4f}\n")

Overall Training:   0%|          | 0/939 [00:00<?, ?it/s]


--- Epoch 1/3 ---
Batch   50/313 | Avg Loss: 1.4790
Batch  100/313 | Avg Loss: 1.1427
Batch  150/313 | Avg Loss: 1.0683
Batch  200/313 | Avg Loss: 1.0765
Batch  250/313 | Avg Loss: 1.0393
Batch  300/313 | Avg Loss: 0.9988

Running Evaluation...
>>> Epoch 1 Accuracy: 0.5750


--- Epoch 2/3 ---
Batch   50/313 | Avg Loss: 0.8643
Batch  100/313 | Avg Loss: 0.8523
Batch  150/313 | Avg Loss: 0.8060
Batch  200/313 | Avg Loss: 0.8166
Batch  250/313 | Avg Loss: 0.8404
Batch  300/313 | Avg Loss: 0.7790

Running Evaluation...
>>> Epoch 2 Accuracy: 0.5790


--- Epoch 3/3 ---
Batch   50/313 | Avg Loss: 0.6368
Batch  100/313 | Avg Loss: 0.5979
Batch  150/313 | Avg Loss: 0.5907
Batch  200/313 | Avg Loss: 0.5971
Batch  250/313 | Avg Loss: 0.5724
Batch  300/313 | Avg Loss: 0.5742

Running Evaluation...
>>> Epoch 3 Accuracy: 0.5850



### Evaluate

Just like how you added an evaluation function to [Trainer](https://huggingface.co/docs/transformers/main/en/main_classes/trainer#transformers.Trainer), you need to do the same when you write your own training loop. But instead of calculating and reporting the metric at the end of each epoch, this time you'll accumulate all the batches with `add_batch` and calculate the metric at the very end.

In [ ]:
!pip install evaluate -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.2 MB/s eta 0:00:00


In [ ]:
import evaluate

metric = evaluate.load("accuracy")
#metric = evaluate.load("f1", "multilabel")
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

{'accuracy': 0.245}

<a id='additional-resources'></a>

## Additional resources

For more fine-tuning examples, refer to:

- [🤗 Transformers Examples](https://github.com/huggingface/transformers/tree/main/examples) includes scripts
  to train common NLP tasks in PyTorch and TensorFlow.

- [🤗 Transformers Notebooks](https://huggingface.co/docs/transformers/main/en/notebooks) contains various notebooks on how to fine-tune a model for specific tasks in PyTorch and TensorFlow.